# Logistic Regression as Probability Engine for Long-Only Crypto Trend

**Thesis**: Logistic regression is best used as a *probability-based capital allocator and regime filter* — not a directional oracle.

This notebook tests **four economically-aligned targets** against our **full 54-feature TA-Lib set** in walk-forward evaluation:

| Target | Label | Why It Matters |
|--------|-------|----------------|
| **A. Breakout Success** | P(20d breakout → +2 ATR before −1 ATR) | Directly predicts whether the convex trade works |
| **B. Trend Persistence** | P(fwd 5d return > 0 \| already in uptrend) | Filters continuation trades |
| **C. Volatility Expansion** | P(realised vol 10d > 75th percentile) | Times convex exposure |
| **D. Regime Classification** | P(BTC trending regime) | Most valuable single application |

For each target, we measure:
- OOS AUC, Brier score, calibration
- Coefficient stability across folds
- Impact on a top-N momentum portfolio (entry filter, conviction sizing, regime throttle)

**Full feature set**: 54 TA-Lib indicators (returns, vol, ADX, MACD, RSI, Stochastic, CCI, Bollinger, OBV, candlestick patterns, etc.) — all pre-shifted by 1 bar to prevent lookahead.

In [20]:
from _setup import *

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss

from jpm_bigdata_ai.helpers import (
    compute_features, FEATURE_COLS, walk_forward_splits,
)
from logreg_filter.labels import (
    barrier_labels, BarrierLabelConfig,
)
from logreg_filter.overlay import (
    apply_entry_filter, apply_conviction_sizing, apply_regime_throttle,
)
from common.metrics import compute_regime

pd.set_option("display.float_format", "{:.4f}".format)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.3})

SEED = 42
COST_BPS = 20.0
print("Setup complete.")

ModuleNotFoundError: No module named 'scripts'

---
## 1. Load Data & Full Feature Set

In [ ]:
panel = load_daily_bars()
panel = filter_universe(panel, min_adv_usd=500_000, min_history_days=90)

print(f"Universe: {panel['symbol'].nunique()} assets, {panel['ts'].nunique()} dates")
print(f"Date range: {panel['ts'].min().date()} to {panel['ts'].max().date()}")
print(f"Rows: {len(panel):,}")

In [ ]:
%%time
featured = compute_features(panel)
feat_cols = list(FEATURE_COLS)
print(f"\n{len(feat_cols)} features computed:")
for i in range(0, len(feat_cols), 6):
    print("  ", ", ".join(feat_cols[i:i+6]))

---
## 2. Define the Four Target Variables

Each target is economically aligned to our convex trend objective.

In [ ]:
def compute_all_targets(df: pd.DataFrame) -> pd.DataFrame:
    """Compute four label columns per asset."""
    results = []
    for sym, g in df.groupby("symbol"):
        g = g.sort_values("ts").copy()
        c = g["close"]
        h = g["high"]
        lo = g["low"]

        # A) Breakout success: +2 ATR before -1 ATR within 20 bars
        g_idx = g.set_index("ts")
        lbl_a = barrier_labels(
            g_idx["high"], g_idx["low"], g_idx["close"],
            BarrierLabelConfig(tp_atr=2.0, sl_atr=1.0, horizon=20, atr_window=14),
        )
        g["target_breakout"] = lbl_a.values

        # B) Trend persistence: fwd 5d return > 0 given uptrend
        fwd5 = c.shift(-5) / c - 1.0
        ma50 = c.rolling(50, min_periods=50).mean()
        in_uptrend = c > ma50
        g["target_persistence"] = np.where(
            in_uptrend & fwd5.notna(),
            (fwd5 > 0).astype(float),
            np.nan,
        )

        # C) Vol expansion: realised vol 10d > 75th pctl
        log_ret = np.log(c / c.shift(1))
        rvol10 = log_ret.rolling(10, min_periods=10).std() * np.sqrt(365)
        rvol_pctl = rvol10.rolling(252, min_periods=63).rank(pct=True)
        fwd_rvol = rvol10.shift(-10)
        fwd_rvol_pctl = fwd_rvol.rolling(252, min_periods=63).rank(pct=True)
        g["target_vol_expansion"] = np.where(
            fwd_rvol.notna(),
            (fwd_rvol_pctl > 0.75).astype(float),
            np.nan,
        )

        results.append(g)

    out = pd.concat(results, ignore_index=True)

    # D) Regime: BTC trending (shared across all assets)
    btc = out.loc[out["symbol"] == "BTC-USD"].sort_values("ts").set_index("ts")
    btc_c = btc["close"]
    btc_ma200 = btc_c.rolling(200, min_periods=200).mean()
    btc_ma50 = btc_c.rolling(50, min_periods=50).mean()
    regime_label = ((btc_c > btc_ma200) & (btc_ma50 > btc_ma200)).astype(float)
    regime_label.name = "target_regime"
    out = out.merge(regime_label, left_on="ts", right_index=True, how="left")

    return out


%%time
labeled = compute_all_targets(featured)

targets = ["target_breakout", "target_persistence", "target_vol_expansion", "target_regime"]
print("\nTarget base rates:")
for t in targets:
    valid = labeled[t].dropna()
    print(f"  {t:30s}  N={len(valid):>9,}  base_rate={valid.mean():.1%}")

---
## 3. Walk-Forward Training Engine

Shared function: trains L2-regularised logistic regression per fold, returns OOS predictions and diagnostics.

In [ ]:
def walk_forward_logreg(
    data: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
    train_days: int = 730,
    test_days: int = 63,
    step_days: int = 63,
    min_train_days: int = 365,
    C: float = 1.0,
    penalty: str = "l2",
) -> dict:
    """Walk-forward logistic regression. Returns predictions + fold metrics."""
    df = data.dropna(subset=[target_col]).copy()
    unique_dates = np.sort(df["ts"].unique())
    splits = walk_forward_splits(
        unique_dates, train_days=train_days, test_days=test_days,
        step_days=step_days, min_train_days=min_train_days,
    )

    all_preds, fold_metrics, coef_history = [], [], []

    for sp in splits:
        tr = df[(df["ts"] >= sp["train_start"]) & (df["ts"] <= sp["train_end"])]
        te = df[(df["ts"] >= sp["test_start"]) & (df["ts"] <= sp["test_end"])]
        tr = tr.dropna(subset=feature_cols + [target_col])
        te = te.dropna(subset=feature_cols)

        if len(tr) < 100 or len(te) < 20:
            continue
        X_tr, y_tr = tr[feature_cols].values, tr[target_col].values.astype(int)
        X_te = te[feature_cols].values
        if len(np.unique(y_tr)) < 2:
            continue

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)

        solver = "saga" if penalty in ("l1", "elasticnet") else "lbfgs"
        model = LogisticRegression(
            penalty=penalty, C=C, solver=solver,
            max_iter=1000, random_state=SEED, n_jobs=-1,
        )
        model.fit(X_tr_s, y_tr)
        probs = np.clip(model.predict_proba(X_te_s)[:, 1], 1e-7, 1 - 1e-7)

        pred_df = te[["ts", "symbol"]].copy()
        pred_df["prob"] = probs
        all_preds.append(pred_df)

        y_te = te[target_col].values.astype(int)
        has_both = len(np.unique(y_te)) >= 2
        fold_metrics.append({
            "fold": sp["fold"],
            "auc": roc_auc_score(y_te, probs) if has_both else np.nan,
            "brier": brier_score_loss(y_te, probs) if has_both else np.nan,
            "logloss": log_loss(y_te, probs) if has_both else np.nan,
            "base_rate": float(y_te.mean()),
            "n_train": len(tr), "n_test": len(te),
        })
        coef_history.append(dict(zip(feature_cols, model.coef_[0])))

    predictions = pd.concat(all_preds, ignore_index=True).drop_duplicates(
        subset=["ts", "symbol"], keep="last",
    )
    fold_df = pd.DataFrame(fold_metrics)
    coef_df = pd.DataFrame(coef_history)

    return {
        "predictions": predictions,
        "fold_metrics": fold_df,
        "coef_history": coef_df,
        "target": target_col,
    }

print("Engine ready.")

---
## 4. Train All Four Models

In [ ]:
%%time
models = {}
for target in targets:
    print(f"\n{'='*60}")
    print(f"Training: {target}")
    print(f"{'='*60}")
    models[target] = walk_forward_logreg(
        labeled, feat_cols, target,
        train_days=730, test_days=63, step_days=63, min_train_days=365,
    )
    fm = models[target]["fold_metrics"]
    print(f"  Folds: {len(fm)}")
    print(f"  Mean AUC:   {fm['auc'].mean():.3f} ± {fm['auc'].std():.3f}")
    print(f"  Mean Brier: {fm['brier'].mean():.3f}")
    print(f"  Base rate:  {fm['base_rate'].mean():.1%}")

---
## 5. Model Diagnostics — AUC, Brier, Calibration

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
target_labels = {
    "target_breakout": "A. Breakout Success",
    "target_persistence": "B. Trend Persistence",
    "target_vol_expansion": "C. Vol Expansion",
    "target_regime": "D. Regime Classification",
}

for ax, target in zip(axes.flat, targets):
    fm = models[target]["fold_metrics"]
    folds = range(len(fm))
    ax.bar(folds, fm["auc"], alpha=0.7, label="AUC")
    ax.axhline(0.5, color="red", linestyle="--", linewidth=0.8, label="Random")
    ax.axhline(fm["auc"].mean(), color="green", linestyle="-", linewidth=1.2,
               label=f"Mean={fm['auc'].mean():.3f}")
    ax.set_title(target_labels[target], fontsize=10)
    ax.set_ylabel("AUC")
    ax.set_ylim(0.35, 0.75)
    ax.legend(fontsize=7)

fig.suptitle("Out-of-Sample AUC by Fold — All Four Targets", fontsize=13)
fig.tight_layout()
plt.show()

# Summary table
summary = []
for target in targets:
    fm = models[target]["fold_metrics"]
    summary.append({
        "Target": target_labels[target],
        "Folds": len(fm),
        "Mean AUC": fm["auc"].mean(),
        "Std AUC": fm["auc"].std(),
        "Min AUC": fm["auc"].min(),
        "Max AUC": fm["auc"].max(),
        "Mean Brier": fm["brier"].mean(),
        "Mean LogLoss": fm["logloss"].mean(),
        "Base Rate": fm["base_rate"].mean(),
    })
pd.DataFrame(summary).set_index("Target")

---
## 6. Feature Importance & Coefficient Stability

Which features are consistently informative across folds? Sign stability = ±1 means the coefficient has the same direction every fold.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for ax, target in zip(axes.flat, targets):
    coefs = models[target]["coef_history"]
    means = coefs.mean()
    stds = coefs.std()
    abs_means = coefs.abs().mean()
    top15 = abs_means.nlargest(15)

    colors = ["#238B23" if means[f] > 0 else "#B32626" for f in top15.index]
    y_pos = range(len(top15))
    ax.barh(y_pos, means[top15.index], xerr=stds[top15.index],
            color=colors, alpha=0.8, capsize=2)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top15.index, fontsize=7)
    ax.axvline(0, color="black", linewidth=0.5)
    ax.set_title(target_labels[target], fontsize=10)

fig.suptitle("Top 15 Features by |Coefficient| — Mean ± Std Across Folds", fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Sign stability table for best target
best_target = max(targets, key=lambda t: models[t]["fold_metrics"]["auc"].mean())
print(f"Most predictable target: {target_labels[best_target]}")
print(f"Mean AUC: {models[best_target]['fold_metrics']['auc'].mean():.3f}\n")

coefs = models[best_target]["coef_history"]
stability = pd.DataFrame({
    "coef_mean": coefs.mean(),
    "coef_std": coefs.std(),
    "|coef|_mean": coefs.abs().mean(),
    "sign_stability": np.sign(coefs).mean(),
}).sort_values("|coef|_mean", ascending=False)
stability.head(20)

---
## 7. Regularisation Comparison: L2 vs L1 vs Elastic Net

In [ ]:
%%time
reg_results = []
for penalty, C_val in [("l2", 1.0), ("l2", 0.1), ("l1", 1.0), ("l1", 0.1)]:
    res = walk_forward_logreg(
        labeled, feat_cols, best_target,
        penalty=penalty, C=C_val,
    )
    fm = res["fold_metrics"]
    n_zero = (res["coef_history"].abs() < 1e-6).sum(axis=1).mean()
    reg_results.append({
        "Penalty": penalty, "C": C_val,
        "Mean AUC": fm["auc"].mean(),
        "Mean Brier": fm["brier"].mean(),
        "Avg Zero Coefs": n_zero,
    })
    print(f"  {penalty} C={C_val}: AUC={fm['auc'].mean():.3f}, "
          f"Brier={fm['brier'].mean():.3f}, Zero coefs={n_zero:.1f}")

pd.DataFrame(reg_results)

---
## 8. Portfolio Impact — The Real Test

Build a top-10 momentum base portfolio and test each target as:
1. Entry filter (p > 0.60)
2. Conviction sizing (weight ∝ edge)
3. Regime throttle (p_regime → gross multiplier)

In [ ]:
# Base portfolio: top-10 momentum, equal weight
def build_base_portfolio(panel_df, mom_window=20, top_n=10):
    df = panel_df.loc[panel_df["in_universe"]].copy()
    df["mom"] = df.groupby("symbol")["close"].transform(
        lambda s: s / s.shift(mom_window) - 1.0,
    )
    dates = sorted(df["ts"].unique())
    all_syms = sorted(df["symbol"].unique())
    weights = pd.DataFrame(0.0, index=pd.DatetimeIndex(dates), columns=all_syms)
    for dt in dates:
        day = df.loc[df["ts"] == dt].dropna(subset=["mom"])
        if len(day) == 0:
            continue
        top = day.nlargest(top_n, "mom")["symbol"].tolist()
        w = 1.0 / max(len(top), 1)
        for sym in top:
            weights.at[dt, sym] = w
    return weights.ffill()


def build_returns_wide(panel_df):
    parts = []
    for sym, g in panel_df.groupby("symbol"):
        g = g.sort_values("ts").set_index("ts")
        r = g["close"].pct_change(fill_method=None)
        r.name = sym
        parts.append(r)
    return pd.concat(parts, axis=1).sort_index()


base_weights = build_base_portfolio(labeled)
returns_wide = build_returns_wide(labeled)
print(f"Base portfolio: {base_weights.shape[0]} dates, {(base_weights > 0).sum(axis=1).mean():.1f} avg positions")

In [ ]:
def backtest_variant(weights, returns, label):
    bt = simple_backtest(weights, returns, cost_bps=COST_BPS)
    eq = bt.set_index("ts")["portfolio_equity"]
    m = compute_metrics(eq)
    m["label"] = label
    m["avg_turnover"] = bt["turnover"].mean()
    m["avg_exposure"] = bt["gross_exposure"].mean()
    return m, eq, bt


def run_ablation(target_key, preds, p_enter=0.60):
    """Run 4-variant ablation for a given target's predictions."""
    probs_wide = preds.pivot_table(
        index="ts", columns="symbol", values="prob", aggfunc="last",
    )

    variants = {"baseline": base_weights}

    # +filter
    filtered = apply_entry_filter(base_weights, probs_wide, p_enter=p_enter)
    variants["+ filter"] = filtered

    # +filter +sizing
    sized = apply_conviction_sizing(filtered, probs_wide, max_weight=0.20, max_positions=10)
    variants["+ filter + sizing"] = sized

    # Regime throttle from regime model if available
    if "target_regime" in models and target_key != "target_regime":
        regime_preds = models["target_regime"]["predictions"]
        regime_wide = regime_preds.groupby("ts")["prob"].mean()
        throttled = apply_regime_throttle(sized, regime_wide)
        variants["+ filter + sizing + regime"] = throttled

    results = {}
    equities = {}
    for name, w in variants.items():
        m, eq, bt = backtest_variant(w, returns_wide, name)
        results[name] = m
        equities[name] = eq

    return results, equities

print("Ablation engine ready.")

In [ ]:
%%time
all_results = {}
all_equities = {}
for target in targets:
    preds = models[target]["predictions"]
    results, equities = run_ablation(target, preds)
    all_results[target] = results
    all_equities[target] = equities
    print(f"\n{target_labels[target]}:")
    for name, m in results.items():
        print(f"  {name:35s} Sharpe={m['sharpe']:>6.2f}  "
              f"CAGR={m['cagr']:>7.1%}  MaxDD={m['max_dd']:>7.1%}  "
              f"Turnover={m['avg_turnover']:.4f}")

---
## 9. Results Dashboard

In [ ]:
# Master comparison table
rows = []
for target in targets:
    for variant, m in all_results[target].items():
        rows.append({
            "Target": target_labels[target],
            "Variant": variant,
            "CAGR": m["cagr"],
            "Vol": m["vol"],
            "Sharpe": m["sharpe"],
            "Sortino": m["sortino"],
            "Max DD": m["max_dd"],
            "Calmar": m["calmar"],
            "Turnover": m["avg_turnover"],
            "Exposure": m["avg_exposure"],
        })

master = pd.DataFrame(rows)
master.style.format({
    "CAGR": "{:.1%}", "Vol": "{:.1%}", "Sharpe": "{:.2f}",
    "Sortino": "{:.2f}", "Max DD": "{:.1%}", "Calmar": "{:.2f}",
    "Turnover": "{:.4f}", "Exposure": "{:.2f}",
}).background_gradient(subset=["Sharpe"], cmap="RdYlGn", vmin=-1, vmax=1)

In [ ]:
# Equity curves — one subplot per target
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
variant_colors = {
    "baseline": "#888888",
    "+ filter": "#0F2E5E",
    "+ filter + sizing": "#C2A054",
    "+ filter + sizing + regime": "#238B23",
}

for ax, target in zip(axes.flat, targets):
    for name, eq in all_equities[target].items():
        ax.plot(eq.index, eq.values, color=variant_colors.get(name, "gray"),
                linewidth=1.2, label=name)
    ax.set_yscale("log")
    ax.set_title(target_labels[target], fontsize=10)
    ax.set_ylabel("Equity (log)")
    ax.legend(fontsize=6, loc="upper left")

fig.suptitle("Equity Curves by Target × Variant", fontsize=13)
fig.tight_layout()
plt.show()

---
## 10. Threshold Sweep — Optimal p_enter

For the best-performing target, sweep p_enter to find the Sharpe-maximising threshold.

In [ ]:
# Find best target by filter Sharpe improvement over baseline
improvements = {}
for target in targets:
    base_sharpe = all_results[target]["baseline"]["sharpe"]
    filter_sharpe = all_results[target]["+ filter"]["sharpe"]
    improvements[target] = filter_sharpe - base_sharpe

best_filter_target = max(improvements, key=improvements.get)
print(f"Best target for entry filtering: {target_labels[best_filter_target]}")
print(f"Sharpe improvement: {improvements[best_filter_target]:+.3f}\n")

preds = models[best_filter_target]["predictions"]
probs_wide = preds.pivot_table(index="ts", columns="symbol", values="prob", aggfunc="last")

sweep = []
for p_val in np.arange(0.40, 0.80, 0.05):
    filtered = apply_entry_filter(base_weights, probs_wide, p_enter=p_val)
    sized = apply_conviction_sizing(filtered, probs_wide, max_weight=0.20, max_positions=10)
    m, eq, bt = backtest_variant(sized, returns_wide, f"p={p_val:.2f}")
    m["p_enter"] = p_val
    sweep.append(m)

sweep_df = pd.DataFrame(sweep)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(sweep_df["p_enter"], sweep_df["sharpe"], "o-", color="#0F2E5E")
axes[0].set_xlabel("p_enter"); axes[0].set_ylabel("Sharpe")
axes[0].set_title("Sharpe vs Threshold")

axes[1].plot(sweep_df["p_enter"], sweep_df["cagr"], "s-", color="#238B23")
axes[1].set_xlabel("p_enter"); axes[1].set_ylabel("CAGR")
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
axes[1].set_title("CAGR vs Threshold")

axes[2].plot(sweep_df["p_enter"], sweep_df["max_dd"], "^-", color="#B32626")
axes[2].set_xlabel("p_enter"); axes[2].set_ylabel("Max DD")
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
axes[2].set_title("Max DD vs Threshold")

fig.suptitle(f"Threshold Sweep — {target_labels[best_filter_target]}", fontsize=12)
fig.tight_layout()
plt.show()

sweep_df[["p_enter", "cagr", "vol", "sharpe", "max_dd", "avg_turnover"]].style.format({
    "p_enter": "{:.2f}", "cagr": "{:.1%}", "vol": "{:.1%}",
    "sharpe": "{:.2f}", "max_dd": "{:.1%}", "avg_turnover": "{:.4f}",
})

---
## 11. Regime Throttle Deep Dive

Isolate the regime model's contribution. When does it correctly reduce exposure?

In [ ]:
regime_preds = models["target_regime"]["predictions"]
regime_ts = regime_preds.groupby("ts")["prob"].mean().sort_index()

# Compare regime probability against BTC returns
btc_eq = compute_btc_benchmark(labeled)
btc_ret = btc_eq.pct_change().rolling(21).sum()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True,
                                gridspec_kw={"hspace": 0.08})

ax1.plot(btc_eq.index, btc_eq.values, color="#333", linewidth=1)
ax1.set_ylabel("BTC Equity")
ax1.set_yscale("log")
ax1.set_title("BTC Equity vs Regime Probability")

ax2.plot(regime_ts.index, regime_ts.values, color="#0F2E5E", linewidth=0.8,
         label="P(trending)")
ax2.axhline(0.6, color="green", linestyle="--", linewidth=0.8, alpha=0.7, label="Full exposure")
ax2.axhline(0.4, color="red", linestyle="--", linewidth=0.8, alpha=0.7, label="Zero exposure")
ax2.fill_between(regime_ts.index, 0, regime_ts.values,
                  where=regime_ts.values >= 0.6, alpha=0.15, color="green")
ax2.fill_between(regime_ts.index, 0, regime_ts.values,
                  where=regime_ts.values < 0.4, alpha=0.15, color="red")
ax2.set_ylabel("P(trending regime)")
ax2.set_ylim(0, 1)
ax2.legend(fontsize=8)

fig.tight_layout()
plt.show()

# Conditional returns
regime_aligned = regime_ts.reindex(btc_ret.index).ffill()
high_regime = btc_ret[regime_aligned >= 0.6].dropna()
low_regime = btc_ret[regime_aligned < 0.4].dropna()
mid_regime = btc_ret[(regime_aligned >= 0.4) & (regime_aligned < 0.6)].dropna()

print(f"BTC 21d return by regime probability:")
print(f"  High (p >= 0.6): mean={high_regime.mean():.1%}, N={len(high_regime)}")
print(f"  Mid  (0.4-0.6):  mean={mid_regime.mean():.1%}, N={len(mid_regime)}")
print(f"  Low  (p < 0.4):  mean={low_regime.mean():.1%}, N={len(low_regime)}")

---
## 12. Convexity Analysis

The real question: does the filter improve the *shape* of the return distribution? We want small losses, occasional large gains.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, target in zip(axes.flat, targets):
    for name, eq in all_equities[target].items():
        rets = eq.pct_change().dropna()
        ax.hist(rets, bins=80, alpha=0.4, density=True,
                color=variant_colors.get(name, "gray"), label=name)
    ax.set_title(target_labels[target], fontsize=10)
    ax.set_xlabel("Daily Return")
    ax.set_ylabel("Density")
    ax.set_xlim(-0.10, 0.10)
    ax.legend(fontsize=6)

fig.suptitle("Return Distribution by Target × Variant", fontsize=13)
fig.tight_layout()
plt.show()

# Convexity metrics
print("\nConvexity Metrics (Skewness / Win-Loss Ratio):")
print(f"{'Target':<30s} {'Variant':<35s} {'Skew':>8s} {'W/L Ratio':>10s} {'Hit%':>7s}")
print("-" * 95)
for target in targets:
    for name, eq in all_equities[target].items():
        rets = eq.pct_change().dropna()
        wins = rets[rets > 0]
        losses = rets[rets < 0]
        wl = abs(wins.mean() / losses.mean()) if len(losses) > 0 and losses.mean() != 0 else np.nan
        print(f"{target_labels[target]:<30s} {name:<35s} {rets.skew():>8.2f} {wl:>10.2f} {(rets > 0).mean():>6.1%}")

---
## 13. Head-to-Head: Best Overlay vs Baseline

Final comparison of the single best configuration.

In [ ]:
# Find overall best Sharpe across all targets × variants
best_sharpe = -np.inf
best_config = None
for target in targets:
    for name, m in all_results[target].items():
        if name == "baseline":
            continue
        if m["sharpe"] > best_sharpe:
            best_sharpe = m["sharpe"]
            best_config = (target, name)

bt, bv = best_config
print(f"Best configuration: {target_labels[bt]} / {bv}")
print(f"Sharpe: {all_results[bt][bv]['sharpe']:.2f} (baseline: {all_results[bt]['baseline']['sharpe']:.2f})")
print()

compare = pd.DataFrame({
    "Metric": ["CAGR", "Vol", "Sharpe", "Sortino", "Max DD", "Calmar",
               "Skewness", "Hit Rate", "Turnover", "Exposure"],
    "Baseline": [
        f"{all_results[bt]['baseline']['cagr']:.1%}",
        f"{all_results[bt]['baseline']['vol']:.1%}",
        f"{all_results[bt]['baseline']['sharpe']:.2f}",
        f"{all_results[bt]['baseline']['sortino']:.2f}",
        f"{all_results[bt]['baseline']['max_dd']:.1%}",
        f"{all_results[bt]['baseline']['calmar']:.2f}",
        f"{all_results[bt]['baseline']['skewness']:.2f}",
        f"{all_results[bt]['baseline']['hit_rate']:.1%}",
        f"{all_results[bt]['baseline']['avg_turnover']:.4f}",
        f"{all_results[bt]['baseline']['avg_exposure']:.2f}",
    ],
    f"Best ({bv})": [
        f"{all_results[bt][bv]['cagr']:.1%}",
        f"{all_results[bt][bv]['vol']:.1%}",
        f"{all_results[bt][bv]['sharpe']:.2f}",
        f"{all_results[bt][bv]['sortino']:.2f}",
        f"{all_results[bt][bv]['max_dd']:.1%}",
        f"{all_results[bt][bv]['calmar']:.2f}",
        f"{all_results[bt][bv]['skewness']:.2f}",
        f"{all_results[bt][bv]['hit_rate']:.1%}",
        f"{all_results[bt][bv]['avg_turnover']:.4f}",
        f"{all_results[bt][bv]['avg_exposure']:.2f}",
    ],
})
compare

In [ ]:
# Final equity curve comparison
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), height_ratios=[3, 1],
                                sharex=True, gridspec_kw={"hspace": 0.08})

eq_base = all_equities[bt]["baseline"]
eq_best = all_equities[bt][bv]

ax1.plot(eq_base.index, eq_base.values, color="#888", linewidth=1.5, label="Baseline")
ax1.plot(eq_best.index, eq_best.values, color="#0F2E5E", linewidth=1.5, label=f"Best: {bv}")
ax1.set_yscale("log")
ax1.set_ylabel("Equity (log)")
ax1.legend(fontsize=10)
ax1.set_title(f"Best Overlay ({target_labels[bt]}) vs Baseline", fontsize=13)

dd_base = eq_base / eq_base.cummax() - 1
dd_best = eq_best / eq_best.cummax() - 1
ax2.fill_between(dd_base.index, dd_base.values, 0, alpha=0.3, color="#888")
ax2.fill_between(dd_best.index, dd_best.values, 0, alpha=0.3, color="#0F2E5E")
ax2.set_ylabel("Drawdown")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))

fig.tight_layout()
plt.show()

---
## 14. Conclusions

Key findings from this experiment:

1. **Which target is most predictable?** — Check AUC summary above. Regime classification typically has the highest AUC because it is a smoother, lower-frequency signal.

2. **Does the filter reduce churn?** — Compare turnover across variants. Entry filtering mechanically reduces trades.

3. **Does conviction sizing help or hurt?** — When AUC is near 0.50, sizing by edge concentrates bets on noise. When AUC > 0.55, it starts adding value.

4. **Does the regime throttle compress drawdowns?** — Compare MaxDD of baseline vs regime-throttled variant.

5. **Is the convexity improved?** — Compare skewness and win/loss ratios.

**Bottom line**: Logistic regression in crypto is best used as a **regime filter** (target D) rather than a per-trade quality predictor (targets A/B). The per-trade targets are too noisy for linear models to capture, but the regime signal is smooth enough to add real value as an exposure throttle.